<a href="https://colab.research.google.com/github/g-poklitar/D704-HW/blob/main/dx704-project-12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DX 704 Week 12 Project

This week's project will revisit the email spam classifier project from week 9 using large language model embeddings instead of custom features.


The full project description and a template notebook are available on GitHub: [Project 12 Materials](https://github.com/bu-cds-dx704/dx704-project-12).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [1]:
!wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

--2026-11-22 16:30:39--  https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip [following]
--2026-11-22 16:30:40--  https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15642124 (15M) [application/zip]
Saving to: ‘enron_spam_data.zip’

enron_spam_data.zip 100%[===================>]  14.92M  --.-KB/s    in 0.09s   

2026-11-22 16:30:40 (160 MB/s) - ‘enron_spam

In [2]:
import pandas as pd

In [3]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip")
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [4]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

np.float64(0.5092834262664611)

## Part 2: Download BERT Model

We will use a pre-trained BERT model to extract embedding vectors as described in lesson 2.1 this week.
Here is sample code to download the model from [Hugging Face](https://huggingface.co/) and extract one vector.
This model is small enough that you can run it with CPU only, but GPUs will be faster if available.

In [5]:
%pip install -U transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 38.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 75.1 MB/s  0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers]


In [6]:
# You may need to install torch and transformers.
# Google Colab will have these installed already.
#
# pip install transformers torch --upgrade

import torch
from transformers import AutoTokenizer, AutoModel

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [8]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.to(device)
bert_model.eval()


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [9]:
@torch.no_grad()
def embed_text(text):
    batch = [text]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
    outputs = bert_model(**inputs)
    # CLS token embedding is the first token's hidden state
    cls_emb = outputs.last_hidden_state[:, 0, :]  # shape: [batch_size, 768]
    return cls_emb.cpu()

In [10]:
v = embed_text("Hi, will you buy my spam?")
v.shape

torch.Size([1, 768])

## Part 3: Create Embedding Vectors

Use BERT to create embeddings for each email in the Enron data set.
You will have to decide how to combine the different columns of the original data set to produce one embedding vector.


Hint: BERT can be run without a GPU, but will be much slower.
Using Google Colab with only a CPU, it runs around 1 embedding per second.
Using Google Colab with the T4 GPU option, it runs around 60 embeddings per second.
Caching is also encouraged to avoid unnecessary reruns.

In [11]:
# YOUR CHANGES HERE

import os
import json
import numpy as np

enron_spam_data["text"] = (
    enron_spam_data["Subject"].fillna("").astype(str)
    + "\n"
    + enron_spam_data["Message"].fillna("").astype(str)
).str.strip()

@torch.no_grad()
def embed_batch(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    outputs = bert_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].float().cpu().numpy()

CACHE_DIR = "embedding_cache"
CHUNK_SIZE = 1024
BATCH_SIZE = 32 if device == "cuda" else 8
os.makedirs(CACHE_DIR, exist_ok=True)

texts = enron_spam_data["text"].tolist()
chunks = []
for start in range(0, len(texts), CHUNK_SIZE):
    path = os.path.join(CACHE_DIR, f"chunk_{start:06d}.npy")
    if os.path.exists(path):
        chunks.append(np.load(path))
        continue
    chunk_texts = texts[start:start + CHUNK_SIZE]
    vecs = np.vstack([embed_batch(chunk_texts[i:i + BATCH_SIZE]) for i in range(0, len(chunk_texts), BATCH_SIZE)])
    np.save(path, vecs)
    chunks.append(vecs)
    print(f"{start + len(chunk_texts)} / {len(texts)}")

embeddings = np.vstack(chunks)
embeddings.shape

1024 / 33716
2048 / 33716
3072 / 33716
4096 / 33716
5120 / 33716
6144 / 33716
7168 / 33716
8192 / 33716
9216 / 33716
10240 / 33716
11264 / 33716
12288 / 33716
13312 / 33716
14336 / 33716
15360 / 33716
16384 / 33716
17408 / 33716
18432 / 33716
19456 / 33716
20480 / 33716
21504 / 33716
22528 / 33716
23552 / 33716
24576 / 33716
25600 / 33716
26624 / 33716
27648 / 33716
28672 / 33716
29696 / 33716
30720 / 33716
31744 / 33716
32768 / 33716
33716 / 33716


(33716, 768)

Save your embeddings in a file "embeddings.tsv.gz" with two columns, Message ID and embedding_vector_json, where embedding_vector_json is a JSON-encoded list.
Make sure that embedding_vector_json is a 1 dimensional list, not 2 dimensional.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

Hint: Gradescope only allows files up to 100MB to be submitted.
Round your embeddings to 3 decimal places to make them smaller.

In [12]:
# YOUR CHANGES HERE

emb_df = pd.DataFrame({
    "Message ID": enron_spam_data["Message ID"],
    "embedding_vector_json": [
        json.dumps(np.round(v.astype(np.float64), 3).tolist(), separators=(",", ":"))
        for v in embeddings
    ],
})
emb_df.to_csv("embeddings.tsv.gz", sep="\t", index=False)
print(os.path.getsize("embeddings.tsv.gz") / 1e6, "MB")

49.279929 MB


Submit "embeddings.tsv.gz" in Gradescope.

## Part 4: Train a Linear Regression

Train an ordinary least squares regression for spam/ham status where spam is treated as target value 1 and ham is treated as target value 0 with your embeddings above as the only input variables.


In [13]:
# YOUR CHANGES HERE

from sklearn.linear_model import LinearRegression

saved = pd.read_csv("embeddings.tsv.gz", sep="\t")
data = saved.merge(enron_spam_data[["Message ID", "Spam/Ham"]], on="Message ID")
X = np.array([json.loads(s) for s in data["embedding_vector_json"]])
y = (data["Spam/Ham"] == "spam").astype(int).to_numpy()

model = LinearRegression(fit_intercept=True)
model.fit(X, y)
print("intercept:", model.intercept_)
print("R^2:", model.score(X, y))
print("accuracy at 0.5 threshold:", ((model.predict(X) >= 0.5).astype(int) == y).mean())

intercept: 1.2879048478465154
R^2: 0.8573737804674544
accuracy at 0.5 threshold: 0.9810179143433385


Save the coefficients of your linear model in a file "coefficients.tsv" with columns dim and coefficient where dim specifies the offset in the embedding vector (0-767).
Don't worry about the bias term (but your model should still have it).

In [14]:
# YOUR CHANGES HERE

coefficients = pd.DataFrame({"dim": range(X.shape[1]), "coefficient": model.coef_})
coefficients.to_csv("coefficients.tsv", sep="\t", index=False)
coefficients

,dim,coefficient
0,0,0.073167
1,1,0.031170
2,2,0.018672
3,3,0.011690
4,4,0.057821
...,...,...
763,763,0.027115
764,764,0.034942
765,765,0.075673
766,766,-0.078351


Submit "coefficients.tsv" in Gradescope.

## Part 5: Search for Relevant Documents

The file "queries.tsv" specifies 10 queries.
For each of the queries, encode them as a vector, and find the message that is closest using $L_2$.

In [15]:
# YOUR CHANGES HERE

queries = pd.read_csv("queries.tsv", sep="\t")
message_ids = data["Message ID"].to_numpy()

rows = []
for query_id, query in zip(queries["query_id"], queries["query"]):
    qv = np.round(embed_batch([query])[0].astype(np.float64), 3)
    distances = np.sqrt(((X - qv) ** 2).sum(axis=1))
    best = int(np.argmin(distances))
    rows.append({
        "query_id": query_id,
        "query": query,
        "query_vector_json": json.dumps(qv.tolist(), separators=(",", ":")),
        "Message ID": int(message_ids[best]),
        "distance": distances[best],
    })

query_matches = pd.DataFrame(rows)
query_matches.merge(enron_spam_data[["Message ID", "Subject"]], on="Message ID")[["query_id", "query", "Message ID", "distance", "Subject"]]

,query_id,query,Message ID,distance,Subject
0,1,accounting arrangements,3273,8.252214,gymnastics pictures
1,2,sales higher than production,2663,7.664062,first delivery atmic marquis
2,3,asked lawyer to write letter about unexpected ...,5057,8.466306,rubbed his hands
3,4,engineering exam,13222,7.851975,credit in 3 d - destruction testing
4,5,discounted tickets,3743,7.842095,holiday e - cards
5,6,unexecuted agreement,21810,6.834887,re [ 4 ]
6,7,well bore,18137,3.965610,hi
7,8,capacity problem,14635,4.377800,central implementation plan
8,9,london partner,14635,4.171547,central implementation plan
9,10,dormant account,15831,7.494073,message subject


Save your results in a file "query-matches.tsv" with columns query_id, query_vector_json, and Message ID.

In [16]:
# YOUR CHANGES HERE

query_matches[["query_id", "query_vector_json", "Message ID"]].to_csv("query-matches.tsv", sep="\t", index=False)

Submit "query-matches.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.